# nanovla on Colab

Clones the repo, installs dependencies, runs the tests and the CPU data smoke test, then benchmarks the LIBERO loader on the GPU runtime.

1. **Runtime → Change runtime type → GPU.**
2. Set `REPO_URL` below. For a private repo, add a `GITHUB_TOKEN` Colab secret (🔑 in the left sidebar).
3. Optional: add an `HF_TOKEN` secret for faster, higher-rate-limit Hub downloads.

Data goes to local disk (`/content/data`), not Drive, because the loader memory-maps its Arrow cache and Drive is too slow for random reads. The full dataset is about 35 GB, so use `keep_raw=False` to avoid storing it twice.

In [ ]:
REPO_URL = "https://github.com/irfanalimd/nanovla.git"  # private: needs a GITHUB_TOKEN Colab secret
BRANCH = "main"
REPO_DIR = "/content/nanovla"
DATA_ROOT = "/content/data/libero"

In [ ]:
import os, subprocess

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

url = REPO_URL
if (tok := secret("GITHUB_TOKEN")) and url.startswith("https://github.com/"):
    url = url.replace("https://", f"https://{tok}@")
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, REPO_DIR)  # instead of `pip install -e .`, which would need a runtime restart

if (hf := secret("HF_TOKEN")):
    os.environ["HF_TOKEN"] = hf
!nvidia-smi -L || echo "no GPU"

## 1. Unit tests (offline, synthetic data) and real-data test

The `--hub` flag also runs the test that downloads the first real shard (~100 MB).

In [ ]:
!python -m pytest -q --hub -p no:warnings

## 2. CPU smoke test on real data

Downloads one shard (episodes 0–2), checks samples, padding and normalization, and times a CPU DataLoader. GPUs are hidden for this step.

In [ ]:
!CUDA_VISIBLE_DEVICES= python -W ignore scripts/smoke_test_data.py --root {DATA_ROOT} --num-workers 2 --preview /content/preview.png
from IPython.display import Image as IPImage, display
display(IPImage("/content/preview.png", width=900))

## 3. Bigger subset + GPU loader benchmark

`NUM_EPISODES = 200` is about 30 shards (~3 GB). Use `None` for all 1693 episodes (35 GB, about 10–20 min to download and convert).

In [ ]:
import time, torch
from torch.utils.data import DataLoader
from nanovla.data import LiberoDataset, LiberoMeta, train_val_episodes

NUM_EPISODES = 200      # None = everything
ACTION_HORIZON = 16
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()

ds_all = LiberoDataset(DATA_ROOT, episodes=None if NUM_EPISODES is None else range(NUM_EPISODES),
                       action_horizon=ACTION_HORIZON, keep_raw=False)
train_eps, val_eps = train_val_episodes(ds_all.meta, val_fraction=0.05, episodes=ds_all.episodes)
train_ds = LiberoDataset(DATA_ROOT, episodes=train_eps, action_horizon=ACTION_HORIZON, progress=False)
val_ds = LiberoDataset(DATA_ROOT, episodes=val_eps, action_horizon=ACTION_HORIZON, progress=False)
print(train_ds); print(val_ds)
print("tasks in train:", len(set(train_ds._task_index.tolist())))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                pin_memory=device == "cuda", persistent_workers=True, drop_last=True, prefetch_factor=4)

it = iter(dl); next(it)  # warm up workers
t0, n, steps = time.perf_counter(), 0, 30
for _ in range(steps):
    b = next(it)
    images = b["images"].to(device, non_blocking=True).float().div_(255)  # [B, cams, 3, 256, 256]
    actions = b["actions"].to(device, non_blocking=True)
    n += images.shape[0]
if device == "cuda":
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f"{n/dt:.0f} samples/s with {NUM_WORKERS} workers ({dt/steps*1e3:.0f} ms/batch of {BATCH_SIZE})")
print({k: (tuple(v.shape), v.dtype) for k, v in b.items() if torch.is_tensor(v)})

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    for c in range(2):
        axes[c, j].imshow(b["images"][j, c].permute(1, 2, 0)); axes[c, j].axis("off")
    axes[0, j].set_title(b["task"][j][:40] + "…", fontsize=8)
plt.tight_layout()

## 4. Training

TODO: add once `nanovla/models` and a training script exist.